# Step 2 · Preview Jev's typed questions

Preview Jev's typed yes/no pathogen and pair-feature questions with the selected pathogen, host, title, and abstract in state. The notebook makes no model or network calls. Jev returns probabilities, not generated JSON, rationale, or evidence text.


## Step 1 · Load the run and prompt/profile definitions

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
import yaml
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'pathogen_atlas_v4').is_dir():
    raise RuntimeError('Open this notebook from the V4 repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'notebooks'))
from _shared import configure_notebook, configured_run_id, staging_root_for_config
from pathogen_atlas_v4.features import resolve_profile
from pathogen_atlas_v4.prompts import build_pathogen_screen_question, build_pair_feature_question
from pathogen_atlas_v4.runs import resolve_run

ROOT = PROJECT_ROOT
CONFIG = yaml.safe_load((ROOT / 'configs/pilot_v1.yaml').read_text(encoding='utf-8'))
RUN_ID = configured_run_id(ROOT)
ROOT, LOG, LOG_PATH = configure_notebook('02_preview_prompts', RUN_ID)
TAXONOMY_ROOT = staging_root_for_config(ROOT, CONFIG) / 'taxonomy'
RUN_TAXONOMY_ROOT = ROOT / 'data/v4/runs' / RUN_ID / 'taxonomy'
if not (RUN_TAXONOMY_ROOT / 'pathogens.parquet').exists():
    raise FileNotFoundError('Run taxonomy was not resolved; run 00_configure_run.ipynb first.')
PATHOGENS = pd.read_parquet(RUN_TAXONOMY_ROOT / 'pathogens.parquet')
HOSTS = pd.read_parquet(RUN_TAXONOMY_ROOT / 'hosts.parquet')
CATALOGUE = yaml.safe_load((ROOT / CONFIG['features']['catalogue']).read_text(encoding='utf-8'))
PROMPT_PATH = ROOT / CONFIG.get('decisions', {}).get('prompt_catalogue', 'configs/prompts.yaml')
PROMPTS = yaml.safe_load(PROMPT_PATH.read_text(encoding='utf-8'))
print('Run:', RUN_ID, '| taxonomy:', TAXONOMY_ROOT.name, '| prompt set:', PROMPTS['prompt_set_version'])
print('Log:', LOG_PATH)
LOG.info('Loaded prompt preview inputs: run=%s taxonomy=%s prompt_set=%s', RUN_ID, TAXONOMY_ROOT.name, PROMPTS['prompt_set_version'])

2026-11-03 10:07 INFO Notebook session started at 2026-11-03T10:07:11.083595+00:00
2026-11-03 10:07 INFO Project root: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4
2026-11-03 10:07 INFO Run log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/02_preview_prompts.log
2026-11-03 10:07 INFO Loaded prompt preview inputs: run=pilot-v1 taxonomy=taxonomy prompt_set=system1-prompt-set-v2


Run: pilot-v1 | taxonomy: taxonomy | prompt set: system1-prompt-set-v2
Log: /Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version4/data/v4/logs/pilot-v1/02_preview_prompts.log


## Step 2 · Resolve one pathogen–host pair to preview

In [2]:
# Leave either ID as None to use the first resolved selection. Set an explicit ID to inspect a different member of the run.
PREVIEW_PATHOGEN_ID = None
PREVIEW_HOST_ID = None
PREVIEW_ARTICLE_TITLE = ''
PREVIEW_ARTICLE_ABSTRACT = ''


host_groups = {key: group.host_id.astype(str).tolist() for key, group in HOSTS.groupby('host_group')}
pathogen_groups = {key: group.pathogen_id.astype(str).tolist() for key, group in PATHOGENS.groupby('pathogen_group')}
resolved = resolve_run(CONFIG, known_host_ids=set(HOSTS.host_id.astype(str)),
    known_pathogen_ids=set(PATHOGENS.pathogen_id.astype(str)),
    host_group_ids=host_groups, pathogen_group_ids=pathogen_groups)
if not resolved.pathogen_ids or not resolved.host_ids:
    raise ValueError('The run must resolve at least one pathogen and one host before prompt preview.')
pathogen_id = str(PREVIEW_PATHOGEN_ID or resolved.pathogen_ids[0])
host_id = str(PREVIEW_HOST_ID or resolved.host_ids[0])
if pathogen_id not in set(resolved.pathogen_ids):
    raise ValueError(f'{pathogen_id} is not in the resolved run pathogen selection.')
if host_id not in set(resolved.host_ids):
    raise ValueError(f'{host_id} is not in the resolved run host selection.')
pathogen = PATHOGENS.set_index('pathogen_id').loc[pathogen_id]
host = HOSTS.set_index('host_id').loc[host_id]
print(f"Pathogen: {pathogen.accepted_name} [{pathogen.pathogen_group}] ({pathogen_id})")
print(f"Host: {host.accepted_name} [{host.host_group}] ({host_id})")
LOG.info('Prompt preview pair selected: pathogen=%s host=%s', pathogen_id, host_id)

2026-11-03 10:07 INFO Prompt preview pair selected: pathogen=ncbi:1972064 host=ncbi:9606


Pathogen: [Olpidiopsis] porphyrae var. koreanae (nom. inval.) [oomycetes] (ncbi:1972064)
Host: Homo sapiens [human] (ncbi:9606)


## Step 3 · Print the pathogen Jev question preview

In [3]:
import textwrap


def print_prompt_readable(prompt, width=80):
    """Print prompt fields with long text wrapped for notebook readability."""
    for key, value in prompt.items():
        if isinstance(value, str):
            print(f"{key}:")
            for paragraph in value.splitlines():
                if paragraph:
                    print(textwrap.fill(
                        paragraph, width=width,
                        initial_indent='  ', subsequent_indent='  '))
                else:
                    print()
        elif isinstance(value, (dict, list)):
            print(f"{key}:")
            rendered = json.dumps(value, indent=2, ensure_ascii=False)
            print(textwrap.indent(rendered, '  '))
        else:
            print(f"{key}: {value}")


pathogen_question = build_pathogen_screen_question(
    pathogen_name=str(pathogen.accepted_name), pathogen_id=pathogen_id,
    host_name=str(host.accepted_name), host_id=host_id,
    article_title=PREVIEW_ARTICLE_TITLE,
    article_abstract=PREVIEW_ARTICLE_ABSTRACT,
    prompt_config=PROMPTS)
print_prompt_readable(pathogen_question)
display(Markdown(f"**Question version:** `{pathogen_question['question_version']}` · **Jev question version:** `{pathogen_question['prompt_version']}`"))
LOG.info('Printed pathogen screening question question_version=%s jev_question=%s', pathogen_question['question_version'], pathogen_question['prompt_version'])


question_version:
  pathogen-screen-system1-v2-threshold-0.7
prompt_version:
  pathogen-screen-noul-v1
state:
  {
    "pathogen": {
      "name": "[Olpidiopsis] porphyrae var. koreanae (nom. inval.)",
      "id": "ncbi:1972064"
    },
    "host": {
      "name": "Homo sapiens",
      "id": "ncbi:9606"
    },
    "article": {
      "title": "",
      "abstract": ""
    }
  }
questions:
  {
    "pathogen_screen": {
      "type": "noul",
      "instructions": "Does `article.abstract` directly establish that `pathogen.name` causes disease in `host.name`? Assess only this abstract and the named pathogen–host pair. Do not count a title-only match, a mere mention, colonization or carriage without disease, a vector or reservoir role, experimental use, or food contamination alone. Do not infer beyond the text.",
      "criteria": {
        "true": "The abstract directly supports disease caused by this named organism in this selected host.",
        "false": "The abstract does not directly suppo

**Question version:** `pathogen-screen-system1-v2-threshold-0.7` · **Jev question version:** `pathogen-screen-noul-v1`

2026-11-03 10:07 INFO Printed pathogen screening question question_version=pathogen-screen-system1-v2-threshold-0.7 jev_question=pathogen-screen-noul-v1


## Step 4 · Print every feature question selected for this pathogen–host pair

The pair and feature profile are resolved from the current run configuration produced by notebook 00. Every applicable question is printed individually with its scope and the host/pathogen context it applies to, wrapped to 80 columns.

In [4]:
profile = resolve_profile(CATALOGUE,
    pathogen_group=str(pathogen.pathogen_group), host_group=str(host.host_group),
    host_taxon_id=str(host.taxon_id), relationship_context='disease',
    profile_version=str(CONFIG['features']['profile_version']))

all_features = profile['ordered_features']
feature_table = pd.DataFrame([{
    'order': index,
    'module': item.get('module', ''),
    'scope': item.get('scope', ''),
    'applies_to': item.get('applies_to_label', ''),
    'feature_id': item.get('feature_id', ''),
    'question_version': item.get('question_version', ''),
    'question': item.get('question', ''),
    'review_status': item.get('review_status', ''),
} for index, item in enumerate(all_features, start=1)])
print('Resolved profile:', profile['profile_id'], profile['profile_version'], profile['resolved_profile_hash'])
print(f'All selected features for {pathogen.accepted_name} × {host.accepted_name}: {len(feature_table)}')
display(feature_table)
LOG.info('Printed all selected pair profile features: count=%d profile=%s', len(feature_table), profile['resolved_profile_hash'])

print(f"\nFeature questions for {pathogen.accepted_name} × {host.accepted_name}: {len(all_features)}")
for index, feature in enumerate(all_features, start=1):
    scope = str(feature.get('scope', ''))
    if scope == 'pathogen':
        entity_context = f"Pathogen: {pathogen.accepted_name}"
    elif scope == 'host':
        entity_context = f"Host: {host.accepted_name}"
    else:
        entity_context = f"Pair: {pathogen.accepted_name} in {host.accepted_name}"
    print(f"\n{index:02d}. {feature.get('label', feature['feature_id'])} [{scope}]")
    print(f"    Feature ID: {feature['feature_id']} · Question version: {feature.get('question_version', '')}")
    print(f"    {entity_context}")
    print(f"    Applies to: {feature.get('applies_to_label', 'Not classified')}")
    print_prompt_readable({'Question': str(feature.get('question', ''))}, width=80)
LOG.info('Printed every selected feature question: count=%d profile=%s',
         len(all_features), profile['resolved_profile_hash'])


Resolved profile: v4-host-aware-feature-questions v4-feature-profile-1.0.2 db87be88130eba3fa88afb3c7a5275d0585e3669da907b80a795ecf14b893b39
All selected features for [Olpidiopsis] porphyrae var. koreanae (nom. inval.) × Homo sapiens: 79


,order,module,scope,applies_to,feature_id,question_version,question,review_status
0,1,core,pathogen_host_pair,All pathogen–host disease pairs,abstract_reports_disease,disease-report-v1,Does this abstract report disease in the speci...,pilot
1,2,pathogen_group:oomycetes,pathogen,All oomycetes,oomycete_zoospores,v4-feature-v1,"In this abstract, is the following claim expli...",curated-draft
2,3,pathogen_group:oomycetes,pathogen,All oomycetes,oomycete_sporangia,v4-feature-v1,"In this abstract, is the following claim expli...",curated-draft
3,4,pathogen_group:oomycetes,pathogen,All oomycetes,oomycete_oospores,v4-feature-v1,"In this abstract, is the following claim expli...",curated-draft
4,5,host_group:human,pathogen_host_pair,"Human · bacteria, fungi, oomycetes, viruses",airborne,human-feature-v1,"In this abstract, is the following claim expli...",inherited-human-catalogue
...,...,...,...,...,...,...,...,...
74,75,pair_context:disease,pathogen_host_pair,All pathogen–host disease pairs,pair_diagnostic_specimen,v4-feature-v1,"In this abstract, is the following claim expli...",curated-draft
75,76,pair_context:disease,pathogen_host_pair,All pathogen–host disease pairs,pair_tissue_tropism,v4-feature-v1,"In this abstract, is the following claim expli...",curated-draft
76,77,pair_context:disease,pathogen_host_pair,All pathogen–host disease pairs,pair_immune_evasion,v4-feature-v1,"In this abstract, is the following claim expli...",curated-draft
77,78,pair_context:disease,pathogen_host_pair,All pathogen–host disease pairs,pair_virulence_determinant,v4-feature-v1,"In this abstract, is the following claim expli...",curated-draft


2026-11-03 10:07 INFO Printed all selected pair profile features: count=79 profile=db87be88130eba3fa88afb3c7a5275d0585e3669da907b80a795ecf14b893b39
2026-11-03 10:07 INFO Printed every selected feature question: count=79 profile=db87be88130eba3fa88afb3c7a5275d0585e3669da907b80a795ecf14b893b39



Feature questions for [Olpidiopsis] porphyrae var. koreanae (nom. inval.) × Homo sapiens: 79

01. Disease relationship established [pathogen_host_pair]
    Feature ID: abstract_reports_disease · Question version: disease-report-v1
    Pair: [Olpidiopsis] porphyrae var. koreanae (nom. inval.) in Homo sapiens
    Applies to: All pathogen–host disease pairs
Question:
  Does this abstract report disease in the specified host caused by the
  specified pathogen?

02. Zoospore production [pathogen]
    Feature ID: oomycete_zoospores · Question version: v4-feature-v1
    Pathogen: [Olpidiopsis] porphyrae var. koreanae (nom. inval.)
    Applies to: All oomycetes
Question:
  In this abstract, is the following claim explicitly reported for the specified
  scope: “the named oomycete produces motile zoospores”?

03. Sporangia [pathogen]
    Feature ID: oomycete_sporangia · Question version: v4-feature-v1
    Pathogen: [Olpidiopsis] porphyrae var. koreanae (nom. inval.)
    Applies to: All oomycete

## Review notes

Step 4 prints every feature question resolved for the current run-selected pathogen and host. It does not make model calls or score abstracts. To preview a different pair within the configured run, set an explicit ID in Step 2.